Protocolo vigente: una trayectoria de 100 generaciones de carácter evolutivo (algoritmo evolutivo poblacional), sin diseño base. Luz 20 m, alturas y secciones variables. Objetivos: masa mínima, utilización promedio máxima y desplazamiento Z mínimo. Criterio principal L/700; transferencia cronológica complementaria. Los resultados previos deben regenerarse tras el entrenamiento.


# 04 — Resultados finales y conclusiones defendibles

**Pregunta:** ¿qué aprendimos y hasta dónde alcanza la evidencia?

`make final` congela decisiones y evalúa prueba. Este notebook solo lee resultados existentes; no evalúa prueba ni ajusta modelos.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.informe_estudio import read_object
from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
STUDY_DIR, REPORT_DIR, CONFIG = context.study_dir, context.report_dir, context.config
DATA_DIR, STATE, RECORDS = CONFIG.data_dir, context.state, context.records
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` · **etiqueta:** `{CONFIG.label}` · **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica breve: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Comprobar el estado del cierre

El manifiesto congelado identifica modelos, políticas, datos y protocolo. La familia se eligió con validación; una familia distinta puede obtener menor error en prueba sin cambiar esa elección. Las nuevas campañas sobre esta prueba ya observada son análisis posteriores.


In [ ]:
display(
    pd.Series(
        {
            "tareas_completas": len(RECORDS),
            "prueba_iniciada": STATE["test_started"],
            "etiqueta": CONFIG.label,
            "diseños_de_prueba": pd.read_csv(DATA_DIR / "test.csv").shape[0],
        }
    )
)
if STATE["test_started"]:
    frozen = read_object(STUDY_DIR / "freeze.json")
    display(
        pd.Series(
            {
                "fecha_de_cierre": frozen["frozen_at_utc"],
                "predictor_congelado": frozen["winner"],
                "firma_de_datos": frozen["dataset_signature"],
            }
        )
    )
else:
    display(
        Markdown(
            "**Prueba todavía no evaluada.** Revisar validación antes de ejecutar `make final`."
        )
    )

## 2. Responder los objetivos con las tablas guardadas

La comparación predictiva usa MAE normalizado. La comparación de decisión usa AUC y calidad final bajo presupuestos comunes. Las medias se acompañan de su variación descriptiva; no se presentan como pruebas de superioridad poblacional.


In [ ]:
families = pd.read_csv(REPORT_DIR / "family_comparison.csv")
display(families)
selection = pd.read_csv(REPORT_DIR / "selection_per_seed.csv")
split_name = "test" if STATE["test_started"] else "validation"
summary = (
    selection.loc[selection["split"].eq(split_name)]
    .groupby(["strategy", "budget"])[
        ["normalized_auc", "relative_hypervolume", "target_reached"]
    ]
    .mean()
)
display(summary)
summary["normalized_auc"].unstack("strategy").plot.bar(
    figsize=(10, 4), ylabel="AUC normalizada", title=f"Selección: {split_name}"
)
plt.tight_layout()
plt.show()

## 3. Interpretar sin forzar un ganador

Las conclusiones se redactan desde el estudio identificado arriba. El mejor predictor por validación puede no encabezar prueba; eso no justifica cambiar la selección congelada. RL puede quedar por debajo de greedy y la ablación puede no mostrar ventaja de usar predicciones.

Los artefactos antiguos quedaron fuera del flujo activo; su síntesis está en la metodología. Compartían los tres objetivos, pero tenían otros presupuestos y validación; no se mezclan sus métricas con esta campaña. Las comprobaciones intermedias de dos objetivos quedaron descartadas.


In [ ]:
display(Markdown((REPORT_DIR / "REPORT.md").read_text(encoding="utf-8")))

## 4. Límites y trabajo pendiente

- El resultado describe diseños registrados y una partición fija; no demuestra generación de nuevas topologías.
- Las categorías raras tienen muy pocos diseños. Cinco semillas no compensan esa falta de datos.
- La utilización promedio no certifica cada barra. No hay registros con utilización promedio mayor que uno para estudiar ese fallo.
- La proximidad entre diseños permite interpolación y limita las afirmaciones de extrapolación.
- Material confirmado: acero A36, fluencia de 250 MPa y comprobaciones según Eurocódigo 3. Siguen pendientes software, cargas, apoyos, agregación exacta del desplazamiento y detalles de agrupación.
- No se midió aceleración frente al evaluador original. Sí se registró RSS máxima muestreada; puede perder picos breves y no equivale a memoria exclusiva del modelo.

Los intervalos predictivos remuestrean diseños pareados después de promediar pérdidas entre semillas; los de selección remuestrean bloques de catálogo y semillas de política. Ninguno convierte episodios superpuestos o semillas en nuevos diseños independientes. La campaña completa requiere interpretar las limitaciones y confirmar la procedencia física.

## Reproducción y comunicación

Usar `make notebooks` para obtener HTML, figuras y notebooks con salidas en `reports/`. El material puede contener datos restringidos; su distribución depende de la autorización aplicable. Las fuentes de estos cuatro notebooks permanecen sin salidas guardadas.


## Procedencia de los métodos y límites de las nuevas comparaciones

El archivo `CONTEXT.md`, en la raíz, resume los antecedentes y enlaza los títulos de los artículos directamente a sus DOI. `SCOPE.md` distingue componentes estándar, adaptaciones al catálogo y límites de la evidencia. Inspeccionar documentos o código no equivale a reproducir resultados.

Las campañas nuevas usan Gymnasium, TorchRL y TensorDict para estandarizar componentes de RL, y pymoo para NSGA-II. La proyección de descendientes y la preselección por hipervolumen predicho son adaptaciones propias. Los formatos históricos se retiraron del código y no se les atribuyen los cambios del protocolo actual.

Las respuestas del catálogo permiten comparar consultas verificables en el dataset. El estudio no puede demostrar seguridad por barra, optimización de nuevas geometrías ni aceleración del evaluador físico original. La prueba histórica ya fue observada: las campañas posteriores siguen siendo análisis de robustez sobre datos conocidos.


## Conclusiones vinculadas a preguntas

Responder precisión frente a referencias, estabilidad del ajuste, costo, utilidad de cada predictor y calidad de las estrategias. La superioridad de RL o redes no es una condición de éxito. Los intervalos son descriptivos y el estudio es exploratorio por conocimiento previo de prueba. La utilización promedio y el desplazamiento operativo no acreditan seguridad completa ni estabilidad cinemática.


In [ ]:
display(pd.read_csv(REPORT_DIR / "cost_totals.csv"))
display(
    Markdown(
        "El total de tiempo por tareas incluye intentos y no se suma nuevamente a sus componentes. Las conclusiones físicas siguen condicionadas por los pendientes de procedencia."
    )
)

## Diagnósticos de robustez
Los errores extremos se conservan y se presentan por separado. La cobertura escasa y la procedencia física pendiente limitan las conclusiones; no se retocan modelos a partir de prueba.

In [ ]:
from truss_research.evaluacion.robustez import EVIDENCE_NOTES

display(Markdown("\n\n".join(EVIDENCE_NOTES.values())))
for diagnostic in (
    "error_tail_summary.csv",
    "category_coverage.csv",
    "physical_provenance.csv",
    "predictive_seed_variation.csv",
    "convergence_diagnostics.csv",
):
    diagnostic_path = REPORT_DIR / diagnostic
    if diagnostic_path.is_file():
        display(pd.read_csv(diagnostic_path).head(20))

## Recursos computacionales y costo de uso

El costo de toda la investigación incluye familias, semillas e intentos. Con entrenamiento paralelo, se distingue la suma de duraciones de trabajos del tiempo transcurrido por invocación. La RSS máxima corresponde a un trabajador y no al conjunto simultáneo. Los tiempos de entrenamiento se observan bajo contención; inferencia y selección se comparan secuencialmente. La amortización usa trabajo acumulado de preparación, paga sus dependencias una vez y distingue catálogos independientes de búsquedas repetidas en el mismo catálogo. La carga se paga una vez; no se suman totales y componentes ni máximos de memoria.

In [ ]:
from IPython.display import Image

cost_root = REPORT_DIR / "benchmark"
if cost_root.is_dir():
    display(pd.read_csv(cost_root / "amortization.csv"))
    display(pd.read_csv(cost_root / "research_cost.csv"))
    display(pd.read_csv(cost_root / "campaign_execution.csv"))
    display(pd.read_csv(cost_root / "worker_jobs.csv"))
    for figure_path in sorted(cost_root.glob("prediction_quality_cost.png")):
        display(Image(filename=str(figure_path)))
else:
    display(
        Markdown(
            "Medición pendiente: ejecutar make benchmark y después make report para este estudio."
        )
    )

## Selección adaptativa con incertidumbre: extensión del mismo estudio

Los ajustes guardados de la familia ganadora forman un ensemble. Para cada candidato disponible se calcula su ganancia marginal de hipervolumen con cada miembro, usando solo respuestas ya reveladas. ensemble_mean maximiza la ganancia media y ensemble_ucb añade beta por su desviación muestral. La incertidumbre representa desacuerdo entre semillas; no es una probabilidad calibrada ni una garantía física. Los modelos permanecen congelados.

Esta extensión posterior al cierre conserva catálogos, inicios y presupuestos. La prueba ya fue examinada y sus comparaciones son exploratorias. Los costos de esta ejecución se presentan aparte del benchmark temporal repetido.

La evaluación puede distribuir bloques entre CPU workers sin cambiar las consultas. execution.json registra duración total; las tablas por trabajador separan trabajo acumulado y RSS individual. Los tiempos se observan bajo concurrencia y el RSS sumado puede contar páginas compartidas.


In [ ]:
adaptive_dir = REPORT_DIR / "adaptive"
if adaptive_dir.is_dir():
    display(pd.read_csv(adaptive_dir / "comparison.csv"))
    display(pd.read_csv(adaptive_dir / "paired_intervals.csv"))
    display(pd.read_csv(adaptive_dir / "uncertainty_diagnostics.csv"))
    display(pd.read_csv(adaptive_dir / "shared_costs.csv"))
    display(pd.read_csv(adaptive_dir / "decision_costs.csv"))
    if (adaptive_dir / "execution.json").is_file():
        display(pd.read_json(adaptive_dir / "execution.json", typ="series"))
    for filename in ("worker_jobs.csv", "worker_resources.csv"):
        if (adaptive_dir / filename).is_file():
            display(pd.read_csv(adaptive_dir / filename))
else:
    display(
        Markdown(
            "Extensión opcional pendiente: ejecutar `make adaptive` con el mismo ESTUDIO y reconstruir el informe con `make report`. No requiere reentrenar."
        )
    )

## Alcance del protocolo actualizado

Los resultados deben contrastar precisión, colas, cobertura y costo completo. Un empate o una desventaja frente a greedy sigue siendo evidencia útil. Reentrenar el mismo dataset no crea una validación externa nueva. Utilización promedio y cotas predictivas no certifican seguridad por barra.


In [ ]:
for filename in [
    "interval_summary.csv",
    "chronological_selection.csv",
    "chronological_resource_phases.csv",
    "objective_sensitivity.csv",
]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(Markdown("**" + filename + "**"))
        display(pd.read_csv(path).head(30))

## Eficiencia de datos y costos adicionales

Las curvas comparan cuatro familias con parámetros fijos y subconjuntos anidados. La referencia común se usa solo para evaluar; cada estrategia decide con escalas de su propio ajuste. Los costos incluyen ajustes y evaluación, separados del benchmark repetido.

In [ ]:
from IPython.display import Image

for filename in [
    "learning_scores.csv",
    "learning_selection_summary.csv",
    "analysis_costs.csv",
]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(pd.read_csv(path).head(30))
figure = REPORT_DIR / "learning_curves.png"
if figure.is_file():
    display(Image(filename=str(figure)))

## Utilidad y costo del preentrenamiento

La pregunta es si la familia preentrenada mejora precisión o calidad temprana de selección para estos datos. Las curvas usan los mismos subconjuntos que los comparadores tabulares. Deben considerarse carga, preparación del contexto, inferencia y memoria. El costo externo de preentrenamiento no está medido y el dataset ya examinado no constituye validación externa nueva.


In [ ]:
cost_path = REPORT_DIR / "benchmark" / "predictor_costs.csv"
if cost_path.is_file():
    costs = pd.read_csv(cost_path)
    display(costs.loc[costs.model.isin(["tabpfn", "targetwise"])])

## Sensibilidad al criterio de servicio
L/700 es el criterio principal adoptado. El caso sin límite Z reevalúa las mismas consultas: no corresponde a políticas reentrenadas para ese escenario. Los valores HV con referencias distintas no son directamente comparables.


In [ ]:
path = REPORT_DIR / "serviceability_sensitivity.csv"
if path.is_file():
    service = pd.read_csv(path)
    display(
        service.groupby(
            ["split", "serviceability", "strategy", "budget"], dropna=False
        )[["normalized_auc", "relative_hypervolume", "observed_failures"]].mean()
    )